# Session 11: Governance and AI risk

In the workshop you name the five roles, draft the three policies and score ten risks for your first AI use on Risk Register.xlsx. The data colleague runs the team notebook, which scores and ranks Kitara's twelve risks and lists the weak spots.

**SOLUTION VERSION. Tutor only.** This is the team notebook. Leaders can follow it on the shared screen. Run the cells from the top. Each cell that starts with '# Your turn' is yours to complete. Run it, then run the check cell after it: the check tells you if you are right and what to fix.

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, zipfile, urllib.request
DATA_ZIP_URL = ""   # your tutor may give you a link; leave empty to upload data.zip instead
if not os.path.exists("customers.csv"):
    if DATA_ZIP_URL:
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data ready:", sorted(f for f in os.listdir(".") if f.endswith(".csv")), "and the docs folder")

## Task 1. Score and rate the risks
Read `risk_register.csv`. Add a `score` column (likelihood times impact) and a `band` column holding `low` (below 8), `medium` (8 to 14) or `high` (15 and above).

In [ ]:
import pandas as pd

r = pd.read_csv('risk_register.csv')
r['score'] = r.likelihood * r.impact
r['band'] = pd.cut(r.score, [0, 7, 14, 25], labels=['low', 'medium', 'high'])
print(r[['id', 'score', 'band']].to_string(index=False))

In [ ]:
assert r.set_index('id').score.to_dict()['R01'] == 20, 'R01 has likelihood 4 and impact 5'
assert r.set_index('id').band.astype(str).to_dict()['R02'] == 'high', 'A score of 15 is high'
assert r.set_index('id').band.astype(str).to_dict()['R05'] == 'medium', 'A score of 8 is medium'
print('Task 1 correct')

## Task 2. Rank and count
Store the id of the highest risk in `top_id`, the number of risks in each band in a dictionary `counts` (keys `high`, `medium`, `low`), and the mean score rounded to 2 decimals in `mean_score`.

In [ ]:
top_id = r.sort_values('score', ascending=False).id.iloc[0]
counts = {b: int((r.band == b).sum()) for b in ['high', 'medium', 'low']}
mean_score = round(float(r.score.mean()), 2)
print(top_id, counts, mean_score)

In [ ]:
assert top_id == 'R01', 'The highest score is 20, risk R01'
assert counts == {'high': 3, 'medium': 5, 'low': 4}, 'Count the rows in each band'
assert mean_score == 10.33, 'Use r.score.mean() and round to 2 decimals'
print('Task 2 correct')

## Task 3. Weak spots
Store in `no_control` the ids of risks whose current control starts with `None`, and in `open_high` the ids of high risks whose status is `Open`.

In [ ]:
no_control = r[r.current_control.str.startswith('None')].id.tolist()
open_high = r[(r.score >= 15) & (r.status == 'Open')].id.tolist()
print(no_control, open_high)

In [ ]:
assert no_control == ['R02'], 'Only R02 has the control None'
assert open_high == ['R01', 'R02'], 'R03 is high but its status is Mitigating'
print('Task 3 correct')

## Task 4. Who carries the most risks
Store in `load` the number of risks per owner role, and in `busiest` the role with the most.

In [ ]:
load = r.owner_role.value_counts()
busiest = load.idxmax()
print(busiest, int(load.max()))

In [ ]:
assert busiest == 'Head of IT', 'Head of IT owns R03, R04, R05 and R10'
assert int(load.max()) == 4, 'The busiest owner holds four risks'
print('Task 4 correct')

## Task 5. Gaps in the data sources
Read `sources.csv`. Store in `gaps` the number of sources with a blank owner, steward and sensitivity as a dictionary with keys `owner`, `steward` and `sensitivity`.

In [ ]:
sources = pd.read_csv('sources.csv')
gaps = {c: int(sources[c].isna().sum()) for c in ['owner', 'steward', 'sensitivity']}
print(gaps)

In [ ]:
assert gaps == {'owner': 4, 'steward': 7, 'sensitivity': 5}, 'Count blanks column by column with isna().sum()'
print('Task 5 correct')

## For the leader
Look at the results. Two of Kitara's three high risks are still open and one risk is owned by a council that has no chair. Who chairs yours, and what can it stop before an AI use goes live? Write your answer in two sentences below.